In [ ]:
%pip install ipython-sql sqlalchemy pandas numpy

In [ ]:
import pandas as pd
import numpy as np

def generar_universo(num_sistemas=50, mediciones_por_planeta=10):
    np.random.seed(42) # Para reproducibilidad en clase
    
    # 1. Tabla de Sistemas Estelares (Estrellas)
    ids_estrellas = range(1, num_sistemas + 1)
    masas = np.random.uniform(0.5, 3.0, num_sistemas) # Masas solares
    temperaturas = masas * 5778 + np.random.normal(0, 200, num_sistemas) # Temp. en Kelvin
    
    df_estrellas = pd.DataFrame({
        'id_estrella': ids_estrellas,
        'masa_solar': masas,
        'temperatura_k': temperaturas,
        'tipo_espectral': np.where(masas > 1.5, 'F', np.where(masas > 0.8, 'G', 'K'))
    })
    
    # 2. Tabla de Exoplanetas
    num_planetas = num_sistemas * 3
    ids_planetas = range(1, num_planetas + 1)
    estrella_padre = np.random.choice(ids_estrellas, num_planetas)
    distancias_au = np.random.uniform(0.1, 5.0, num_planetas)
    
    # Aplicar Ley de Kepler para el periodo orbital + un poco de ruido
    df_planetas = pd.DataFrame({
        'id_planeta': ids_planetas,
        'id_estrella': estrella_padre,
        'distancia_au': distancias_au
    })
    
    # Merge temporal para calcular la física
    temp_merge = df_planetas.merge(df_estrellas[['id_estrella', 'masa_solar']], on='id_estrella')
    periodos_teoricos = np.sqrt((temp_merge['distancia_au']**3) / temp_merge['masa_solar'])
    df_planetas['periodo_dias'] = periodos_teoricos * 365.25 + np.random.normal(0, 2, num_planetas)
    
    # 3. Tabla de Mediciones de Tránsito (Ruido estocástico para practicar SQL)
    ids_medicion = range(1, (num_planetas * mediciones_por_planeta) + 1)
    planeta_medido = np.repeat(ids_planetas, mediciones_por_planeta)
    
    # Simulamos mediciones de radio con error instrumental
    radios_base = np.random.uniform(0.5, 15.0, num_planetas)
    radios_medidos = np.repeat(radios_base, mediciones_por_planeta) + np.random.normal(0, 0.5, len(ids_medicion))
    
    df_mediciones = pd.DataFrame({
        'id_medicion': ids_medicion,
        'id_planeta': planeta_medido,
        'radio_terrestre_observado': radios_medidos,
        'calidad_observacion': np.random.choice(['Alta', 'Media', 'Baja'], len(ids_medicion), p=[0.6, 0.3, 0.1])
    })
    
    return df_estrellas, df_planetas, df_mediciones



In [ ]:
# Instanciamos los datos
estrellas, planetas, mediciones = generar_universo()
print("¡Universo generado con éxito!")

In [ ]:
import sqlite3

# Crear conexión en memoria
conn = sqlite3.connect(':memory:')

# Volcar los DataFrames de Pandas a tablas SQL reales
estrellas.to_sql('estrellas', conn, index=False)
planetas.to_sql('planetas', conn, index=False)
mediciones.to_sql('mediciones', conn, index=False)

# Cargar la extensión mágica de Jupyter para SQL
%load_ext sql
%sql sqlite:///:memory:

### 🪐 Ejercicio 1: Conociendo nuestro vecindario estelar

¡Bienvenido al centro de mando del observatorio! Para familiarizarte con la base de datos, tu primera misión es extraer información específica de nuestro catálogo de estrellas.

**Tu reto:** Escribe una consulta en SQL que devuelva:
1. El `id_estrella`, la `masa_solar` y el `tipo_espectral` de **todas las estrellas de tipo espectral 'K'** (estrellas más frías y longevas que nuestro Sol).
2. Ordena los resultados de mayor a menor masa.
3. Limita el resultado a los primeros 5 registros.

*Pista: Recuerda usar `WHERE`, `ORDER BY ... DESC` y `LIMIT`.*

In [ ]:
%%sql
-- Tu código SQL aquí abajo:

### 📊 Ejercicio 2: Estadísticas del Universo

Para entender la demografía de nuestra muestra, necesitamos agrupar los datos. Vamos a calcular cuántas estrellas tenemos de cada tipo y su masa promedio.

**Tu reto:**
Escribe una consulta que agrupe las estrellas por su `tipo_espectral` y calcule:
1. El total de estrellas en cada categoría (nombra la columna como `total_estrellas`).
2. El promedio de su `temperatura_k` (nombra la columna como `temperatura_promedio`).
3. Filtra los resultados para mostrar **solo** aquellos tipos espectrales cuyo promedio de temperatura sea mayor a 5000 Kelvin.

*Pista: Para filtrar después de un `GROUP BY`, debes usar `HAVING` en lugar de `WHERE`.*

%%sql
-- Tu código SQL aquí abajo:

### 🔗 Ejercicio 3: Cruzando Datos Físicos (Kepler a prueba)

Los planetas están en una tabla y sus estrellas en otra. Para comprobar si los planetas más lejanos tardan más en dar la vuelta a su estrella (Tercera Ley de Kepler), necesitamos juntar ambas tablas.

**Tu reto:**
Realiza un `INNER JOIN` entre la tabla `planetas` y la tabla `estrellas` usando la llave que las conecta (`id_estrella`). Selecciona:
1. El `id_planeta`.
2. La `distancia_au` del planeta.
3. El `periodo_dias` del planeta.
4. El `tipo_espectral` de su estrella.
5. Filtra para ver únicamente los planetas que están a más de 2 unidades astronómicas (`distancia_au > 2`).

In [ ]:
%%sql
-- Tu código SQL aquí abajo:

### 🧼 Ejercicio 4: Control de Calidad y Limpieza de Datos

En la tabla `mediciones` tenemos múltiples observaciones del radio de cada planeta, pero algunas tienen ruido instrumental o baja calidad. Queremos el radio promedio real de cada planeta, pero solo usando datos confiables.

**Tu reto:**
Une las tablas `planetas` y `mediciones` para obtener:
1. El `id_planeta`.
2. El promedio del `radio_terrestre_observado` (llámalo `radio_promedio`).
3. Filtra las filas ANTES de agrupar para incluir **solo** las mediciones cuya `calidad_observacion` sea 'Alta' o 'Media'.
4. Muestra los 10 planetas con el radio promedio más grande.

In [ ]:
%%sql
-- Tu código SQL aquí abajo: